# Обучение модели прогноза нагрузки (workload) на горизонте 4 месяцев (months)

Этот ноутбук (notebook):

- загружает `../data/processed/workload_training_dataset.csv`
- делает backtest (backtest) по времени (time) с горизонтом (horizon) 4 месяца (months)
- обучает `CatBoostRegressor (CatBoostRegressor)` как регрессию (regression)
- сравнивает с baseline (baseline) (гибридный (hybrid))
- считает метрики (metrics) по двум истинам (targets):
  - `workload_fact` (DB truth)
  - `workload_manual_pct` (Excel manual) (если присутствует)
- сохраняет предсказания (predictions) в `../data/processed/model_predictions_catboost_v1.csv`



In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor

DATA_PATH = Path("../data/processed/workload_training_dataset.csv")
OUT_PRED_PATH = Path("../data/processed/model_predictions_catboost_v1.csv")

# Горизонт прогноза (forecast horizon)
H = 4

# Минимум истории (history) до первой отсечки (cutoff)
MIN_TRAIN_MONTHS = 18

# Параметры CatBoost (CatBoost parameters)
CATBOOST_PARAMS = dict(
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=200,
)

DATA_PATH


In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["period"])

# Нормализуем базовые поля (basic fields)
df = df.rename(columns={
    "workload_fact": "y",  # DB truth (target)
})

# Убираем строки без target (target)
df = df.dropna(subset=["y", "resist_id", "period"]).copy()

# Упорядочиваем

df = df.sort_values(["resist_id", "period"]).reset_index(drop=True)

# Явные признаки (features)
WHITELIST_KEYS = [
    "workDayMonth_2",
    "hourInWeek_12",
    "effective_13",
    "absenteeism_14",
    "employFact_7",
    "cross_5",
    "overtime_4",
    "sla_15",
]

TIME_FEATURES = ["month_num", "year_num", "quarter_num"]
LAG_FEATURES = ["lag_1", "lag_2", "lag_3", "lag_12"]
ROLL_FEATURES = ["roll_med_6", "roll_mean_6"]

CAT_FEATURES = ["resist_id", "division_title"]

FEATURES = WHITELIST_KEYS + TIME_FEATURES + LAG_FEATURES + ROLL_FEATURES + CAT_FEATURES

missing = [c for c in FEATURES if c not in df.columns]
missing


In [ ]:
def compute_metrics(y_true, y_pred, large_abs=10, large_pct=0.1):
    """Метрики (metrics) для нагрузки (workload).

    large_abs — порог (threshold) в п.п. (percentage points), large_pct — порог в доле от факта.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if len(y_true) == 0:
        return {"MAE": np.nan, "RMSE": np.nan, "MAPE_%": np.nan, "large_error_share": np.nan}

    error = y_pred - y_true
    abs_error = np.abs(error)

    mae = abs_error.mean()
    rmse = np.sqrt((error ** 2).mean())

    mape = np.nan
    nonzero = y_true != 0
    if nonzero.any():
        mape = (abs_error[nonzero] / np.abs(y_true[nonzero]) * 100).mean()

    large_mask = (abs_error > large_abs) | (abs_error > np.abs(y_true) * large_pct)
    large_share = large_mask.mean()

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE_%": mape,
        "large_error_share": large_share,
    }


def hybrid_baseline_by_resist(df_in: pd.DataFrame) -> pd.Series:
    """Hybrid baseline (hybrid baseline) для нагрузки (workload) по resist_id.

    Приоритет:
    year_ago -> rolling_median(6) -> last_value -> global_median
    """
    df0 = df_in.sort_values(["resist_id", "period"]).copy()
    df0["month"] = df0["period"].dt.to_period("M").dt.to_timestamp()

    df0["month_prev"] = df0["month"] - pd.DateOffset(months=12)
    prev = (
        df0[["resist_id", "month", "y"]]
        .rename(columns={"month": "month_prev", "y": "y_prev"})
    )
    df0 = df0.merge(prev, on=["resist_id", "month_prev"], how="left")

    year_ago = df0["y_prev"]
    rolling_med = df0.groupby("resist_id")["y"].transform(lambda s: s.shift(1).rolling(6, min_periods=3).median())
    last_val = df0.groupby("resist_id")["y"].transform(lambda s: s.shift(1))
    global_med = df0.groupby("resist_id")["y"].transform("median")

    baseline = year_ago.copy()
    mask = baseline.isna()
    baseline.loc[mask] = rolling_med.loc[mask]

    mask = baseline.isna()
    baseline.loc[mask] = last_val.loc[mask]

    mask = baseline.isna()
    baseline.loc[mask] = global_med.loc[mask]

    return baseline.clip(lower=0)


# Добавим baseline в df

df["baseline"] = hybrid_baseline_by_resist(df)

compute_metrics(df["y"], df["baseline"])


In [ ]:
def make_cutoffs(df_in: pd.DataFrame, min_train_months: int, horizon: int) -> list[pd.Timestamp]:
    """Список дат отсечки (cutoffs) по месяцам (months)."""
    months = sorted(df_in["period"].dt.to_period("M").dt.to_timestamp().unique())
    if len(months) < (min_train_months + horizon + 1):
        return []
    # отсечки начинаются после min_train_months месяцев истории
    cutoffs = months[min_train_months:]
    # убираем те, где не хватает горизонта вперёд
    cutoffs = [c for c in cutoffs if c + pd.DateOffset(months=horizon) <= months[-1]]
    return cutoffs


def add_horizon_column(df_in: pd.DataFrame, cutoff: pd.Timestamp) -> pd.Series:
    """Горизонт (horizon) в месяцах для каждой строки относительно cutoff."""
    return (
        (df_in["period"].dt.to_period("M").dt.to_timestamp() - cutoff)
        / np.timedelta64(1, "M")
    )


cutoffs = make_cutoffs(df, MIN_TRAIN_MONTHS, H)
len(cutoffs), cutoffs[:3], cutoffs[-3:]


In [ ]:
# Backtest (backtest): обучаем на <= cutoff и предсказываем на (cutoff+1..cutoff+H)

# Категориальные признаки (categorical features) для CatBoost
cat_features = ["resist_id", "division_title"]

# Список признаков (features)
X_cols = FEATURES

all_preds = []
all_scores = []

for cutoff in cutoffs:
    train_mask = df["period"].dt.to_period("M").dt.to_timestamp() <= cutoff
    test_mask = (
        (df["period"].dt.to_period("M").dt.to_timestamp() > cutoff)
        & (df["period"].dt.to_period("M").dt.to_timestamp() <= cutoff + pd.DateOffset(months=H))
    )

    train = df[train_mask].copy()
    test = df[test_mask].copy()

    # Убираем строки без лагов (lags) — минимум lag_1
    train = train.dropna(subset=["lag_1"])
    test = test.dropna(subset=["lag_1"])

    if len(train) == 0 or len(test) == 0:
        continue

    X_train = train[X_cols]
    y_train = train["y"]

    X_test = test[X_cols]
    y_test = test["y"]

    model = CatBoostRegressor(**CATBOOST_PARAMS)
    model.fit(X_train, y_train, cat_features=cat_features)

    y_pred = model.predict(X_test)

    # baseline на тесте
    y_base = test["baseline"].to_numpy(dtype=float)

    # метрики по DB truth
    m_model = compute_metrics(y_test, y_pred)
    m_base = compute_metrics(y_test, y_base)

    all_scores.append({"cutoff": cutoff, "source": "catboost_v1", **m_model})
    all_scores.append({"cutoff": cutoff, "source": "baseline", **m_base})

    # метрики по Excel manual (если есть)
    if "workload_manual_pct" in test.columns:
        manual = test["workload_manual_pct"].to_numpy(dtype=float)
        mask_manual = ~np.isnan(manual)
        if mask_manual.any():
            m_model_m = compute_metrics(manual[mask_manual], y_pred[mask_manual])
            m_base_m = compute_metrics(manual[mask_manual], y_base[mask_manual])
            all_scores.append({"cutoff": cutoff, "source": "catboost_v1_vs_manual", **m_model_m})
            all_scores.append({"cutoff": cutoff, "source": "baseline_vs_manual", **m_base_m})

    # сохраняем предсказания
    test_out = test[["resist_id", "division_title", "period"]].copy()
    test_out["cutoff"] = cutoff
    test_out["y_true"] = y_test.values
    test_out["y_pred"] = y_pred
    test_out["y_baseline"] = y_base
    if "workload_manual_pct" in test.columns:
        test_out["y_manual"] = test["workload_manual_pct"].values
    all_preds.append(test_out)


df_scores = pd.DataFrame(all_scores)
df_preds = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame()

(df_scores.head(), df_scores["source"].value_counts(), df_preds.head(), df_preds.shape)


In [ ]:
# Итоги (summary) и сохранение предсказаний (predictions)

if df_scores.empty:
    print("Нет результатов backtest (backtest): проверь, что хватает истории и что есть lag_1")
else:
    # усреднение по cutoffs
    summary = df_scores.groupby("source")[["MAE", "RMSE", "MAPE_%", "large_error_share"]].mean().sort_values("MAE")
    display(summary)

# Сохраняем предсказания (predictions)
OUT_PRED_PATH.parent.mkdir(parents=True, exist_ok=True)
df_preds.to_csv(OUT_PRED_PATH, index=False)
OUT_PRED_PATH
